<!--
Created by Brad Delatte
Bayou Bytes
azure-sql-schema-compare
Initial architecture and implementation: 2026
-->

# Cell 1 — Notebook Settings
Set the run mode, output file, debug options, and timing options.

In [ ]:
# ------------------------------------------------------------
# NOTEBOOK SETTINGS
# ------------------------------------------------------------

# TABLE_SOURCE_MODE options:
#
# "json" = use tables_to_compare.json
# "sql"  = discover tables automatically
#

# TABLE_SOURCE_MODE = "json"
TABLE_SOURCE_MODE = "sql"


# ------------------------------------------------------------
# OUTPUT SETTINGS
# ------------------------------------------------------------

EXPORT_TO_EXCEL = True

OUTPUT_ROOT = "../output"
MAX_OUTPUT_FOLDERS = 4


# ------------------------------------------------------------
# DEBUG SETTINGS
# ------------------------------------------------------------

SHOW_FIRST_N_TABLES = 20


# ------------------------------------------------------------
# TIMER SETTINGS
# ------------------------------------------------------------

SHOW_TIMING = True

# Cell 2 — Imports
Import required packages and load the reusable SQL connection helper.

In [ ]:
from dotenv import load_dotenv
import pandas as pd
import pyodbc
import json
import os
import warnings
import time
import sys

from datetime import datetime
from pathlib import Path

warnings.filterwarnings(
    "ignore",
    message="pandas only supports SQLAlchemy connectable"
)

sys.path.append(str(Path("../")))

from src.connections import get_conn

print("Imports successful")

# Cell 3 — Start Timer
Record the notebook start time so the full run duration can be calculated at the end.

In [ ]:
start_datetime = datetime.now()
start_time = time.time()

if SHOW_TIMING:

    print(
        "START TIME:",
        start_datetime.strftime("%Y-%m-%d %H:%M:%S")
    )

RUN_ID = start_datetime.strftime("%Y%m%d_%H%M%S")

OUTPUT_FOLDER = Path(OUTPUT_ROOT) / RUN_ID

OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)

SCHEMA_COMPARE_OUTPUT_FILE = OUTPUT_FOLDER / "schema_compare_results.xlsx"

print("RUN_ID:", RUN_ID)
print("OUTPUT_FOLDER:", OUTPUT_FOLDER)

# Cell 4 — Load Database and Table Configuration
Load the database list and either load selected tables from JSON or discover all tables from SQL.

In [ ]:
with open("../config/databases.json", "r") as f:
    DATABASES = json.load(f)

print("DATABASES")
print(DATABASES)


if TABLE_SOURCE_MODE == "json":

    with open("../config/tables_to_compare.json", "r") as f:
        table_config = json.load(f)

    TABLES_TO_COMPARE = [
        (x["schema"], x["table"])
        for x in table_config
    ]


elif TABLE_SOURCE_MODE == "sql":

    # ------------------------------------------------------------
    # Load SQL once
    # ------------------------------------------------------------

    sql_path = Path("../sql/get_base_tables.sql")

    with open(sql_path, "r") as f:
        get_base_tables_sql = f.read()


    # ------------------------------------------------------------
    # Get tables from database
    # ------------------------------------------------------------

    def get_base_tables(database: str):

        with get_conn(database) as conn:

            return pd.read_sql(
                get_base_tables_sql,
                conn
            )

    table_rows = []

    for database in DATABASES:

        df = get_base_tables(database)

        df["database_name"] = database

        table_rows.append(df)

        print(
            f"Loaded table list from "
            f"{database}: "
            f"{len(df)} tables"
        )

    all_tables_found = pd.concat(
        table_rows,
        ignore_index=True
    )

    TABLES_TO_COMPARE = (
        all_tables_found[
            ["schema_name", "table_name"]
        ]
        .drop_duplicates()
        .sort_values(
            ["schema_name", "table_name"]
        )
        .apply(
            lambda row: (
                row["schema_name"],
                row["table_name"]
            ),
            axis=1
        )
        .tolist()
    )

else:

    raise ValueError(
        "TABLE_SOURCE_MODE must be either "
        "'json' or 'sql'"
    )


print("\nTABLES_TO_COMPARE")

print(
    f"Total tables: "
    f"{len(TABLES_TO_COMPARE)}"
)

for table in TABLES_TO_COMPARE[:SHOW_FIRST_N_TABLES]:

    print(table)

if len(TABLES_TO_COMPARE) > SHOW_FIRST_N_TABLES:

    print(
        f"... showing first "
        f"{SHOW_FIRST_N_TABLES} "
        f"of {len(TABLES_TO_COMPARE)} tables"
    )

# Cell 5 — Test Database Connection
Connect to the first configured database and confirm the connection works.

In [ ]:
test_db = DATABASES[0]

with get_conn(test_db) as conn:

    cursor = conn.cursor()

    cursor.execute("SELECT DB_NAME()")

    print(cursor.fetchone()[0])

# Cell 6 — Define Schema Reader
Create a function that reads column names, data types, lengths, nullability, and identity settings for an entire database.

In [ ]:
def get_database_schema(database):

    sql = """
    SELECT
         DB_NAME() AS database_name
        ,s.name AS schema_name
        ,t.name AS table_name
        ,c.column_id
        ,c.name AS column_name
        ,ty.name AS data_type
        ,c.max_length
        ,c.precision
        ,c.scale
        ,c.is_nullable
        ,c.is_identity
    FROM sys.tables t
    JOIN sys.schemas s
        ON t.schema_id = s.schema_id
    JOIN sys.columns c
        ON t.object_id = c.object_id
    JOIN sys.types ty
        ON c.user_type_id = ty.user_type_id
    WHERE t.is_ms_shipped = 0
    ORDER BY
         s.name
        ,t.name
        ,c.column_id;
    """

    with get_conn(database) as conn:
        return pd.read_sql(sql, conn)

# Cell 7 — Define Primary Key Reader
Create a function that reads primary key names and primary key columns for an entire database.

In [ ]:
def get_database_primary_keys(database):

    sql = """
    SELECT
         DB_NAME() AS database_name
        ,s.name AS schema_name
        ,t.name AS table_name
        ,kc.name AS pk_name
        ,c.name AS column_name
        ,ic.key_ordinal
    FROM sys.key_constraints kc
    JOIN sys.tables t
        ON kc.parent_object_id = t.object_id
    JOIN sys.schemas s
        ON t.schema_id = s.schema_id
    JOIN sys.index_columns ic
        ON kc.parent_object_id = ic.object_id
       AND kc.unique_index_id = ic.index_id
    JOIN sys.columns c
        ON ic.object_id = c.object_id
       AND ic.column_id = c.column_id
    WHERE kc.type = 'PK'
    ORDER BY
         s.name
        ,t.name
        ,ic.key_ordinal;
    """

    with get_conn(database) as conn:
        return pd.read_sql(sql, conn)

# Cell 8 — Collect Metadata From All Databases
Load schema and primary key metadata from each database, then filter to the selected table list.

In [ ]:
schema_results = []
pk_results = []

tables_filter_df = pd.DataFrame(
    TABLES_TO_COMPARE,
    columns=["schema_name", "table_name"]
)

for database in DATABASES:

    try:
        print(f"Loading metadata from {database}...")

        schema_df = get_database_schema(database)
        pk_df = get_database_primary_keys(database)

        schema_df = schema_df.merge(
            tables_filter_df,
            on=["schema_name", "table_name"],
            how="inner"
        )

        pk_df = pk_df.merge(
            tables_filter_df,
            on=["schema_name", "table_name"],
            how="inner"
        )

        schema_results.append(schema_df)
        pk_results.append(pk_df)

        print(f"SUCCESS: {database}")
        print(f"  Schema rows: {len(schema_df)}")
        print(f"  PK rows    : {len(pk_df)}")

    except Exception as e:
        print(f"FAILED: {database}")
        print(e)

all_schema = pd.concat(
    schema_results,
    ignore_index=True
)

all_pks = pd.concat(
    pk_results,
    ignore_index=True
)

# Cell 9 — View Raw Schema Results
Display the full collected schema results before comparison.

In [ ]:
display(all_schema)

# Cell 10 — Compare Column Presence
Show which columns exist across the compared databases. Blank cells mean the column is missing from that database.

In [ ]:
column_presence = (
    all_schema
    .pivot_table(
        index=[
            "schema_name",
            "table_name",
            "column_name"
        ],
        columns="database_name",
        values="data_type",
        aggfunc="first"
    )
)

display(column_presence)

# Cell 11 — Find Missing Columns
Find columns that exist in one database table but are missing from another database table.

In [ ]:
all_schema["column_name_norm"] = (
    all_schema["column_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

results = []

master_columns = (
    all_schema
    .groupby(["schema_name", "table_name"])["column_name_norm"]
    .apply(lambda x: sorted(set(x)))
    .reset_index(name="expected_columns")
)

for _, row in master_columns.iterrows():

    schema_name = row["schema_name"]
    table_name = row["table_name"]
    expected_columns = set(row["expected_columns"])

    table_rows = all_schema[
        (all_schema["schema_name"] == schema_name)
        &
        (all_schema["table_name"] == table_name)
    ]

    for database_name, db_group in table_rows.groupby("database_name"):

        actual_columns = set(db_group["column_name_norm"])

        missing_columns = expected_columns - actual_columns

        for missing_col in sorted(missing_columns):

            databases_that_have_it = sorted(
                table_rows[
                    table_rows["column_name_norm"] == missing_col
                ]["database_name"].unique()
            )

            results.append({
                "database_missing_column": database_name,
                "schema_name": schema_name,
                "table_name": table_name,
                "missing_column": missing_col,
                "databases_that_have_column": ", ".join(databases_that_have_it)
            })

missing_columns_detail = pd.DataFrame(results)

display(missing_columns_detail)

# Cell 12 — Detect Schema Mismatches
Find columns where data type, length, precision, scale, nullability, or identity settings do not match.

In [ ]:
mismatches = (
    all_schema
    .groupby([
        "schema_name",
        "table_name",
        "column_name"
    ])
    .agg(
        data_type_count=("data_type", "nunique"),
        nullable_count=("is_nullable", "nunique"),
        identity_count=("is_identity", "nunique"),
        max_length_count=("max_length", "nunique"),
        precision_count=("precision", "nunique"),
        scale_count=("scale", "nunique")
    )
    .reset_index()
)

mismatches = mismatches[
    (mismatches["data_type_count"] > 1)
    |
    (mismatches["nullable_count"] > 1)
    |
    (mismatches["identity_count"] > 1)
    |
    (mismatches["max_length_count"] > 1)
    |
    (mismatches["precision_count"] > 1)
    |
    (mismatches["scale_count"] > 1)
]

display(mismatches)

# Cell 13 — View Detailed Mismatch Records
Show the database-level details for each mismatched column.

In [ ]:
mismatch_detail = all_schema.merge(

    mismatches[
        [
            "schema_name",
            "table_name",
            "column_name"
        ]
    ],

    on=[
        "schema_name",
        "table_name",
        "column_name"
    ],

    how="inner"
)

mismatch_detail = mismatch_detail.sort_values(
    [
        "schema_name",
        "table_name",
        "column_name",
        "database_name"
    ]
)

display(mismatch_detail)

# Cell 14 — Compare Primary Keys
Display primary key names and primary key columns across the compared databases.

In [ ]:
display(all_pks)

# Cell 15 — Export Results and End Timer
Export all comparison results to Excel and print the total runtime.

In [ ]:
if EXPORT_TO_EXCEL:

    with pd.ExcelWriter(
        SCHEMA_COMPARE_OUTPUT_FILE,
        engine="openpyxl"
    ) as writer:

        all_schema.to_excel(
            writer,
            sheet_name="all_schema",
            index=False
        )

        all_pks.to_excel(
            writer,
            sheet_name="primary_keys",
            index=False
        )

        column_presence.to_excel(
            writer,
            sheet_name="column_presence"
        )

        missing_columns_detail.to_excel(
            writer,
            sheet_name="missing_columns",
            index=False
        )

        mismatches.to_excel(
            writer,
            sheet_name="datatype_mismatches",
            index=False
        )

        mismatch_detail.to_excel(
            writer,
            sheet_name="mismatch_detail",
            index=False
        )
    
        metadata_df = pd.DataFrame([
            {
                "author": "Brad Delatte",
                "tool": "azure-sql-schema-compare",
                "created": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            }
        ])

        metadata_df.to_excel(
            writer,
            sheet_name="_metadata",
            index=False
        )

        writer.sheets["_metadata"].sheet_state = "hidden"

    print(f"\nSaved: {SCHEMA_COMPARE_OUTPUT_FILE}")


end_datetime = datetime.now()
end_time = time.time()

elapsed_seconds = end_time - start_time
elapsed_minutes = elapsed_seconds / 60

output_root = Path(OUTPUT_ROOT)

run_folders = sorted(
    [
        p for p in output_root.iterdir()
        if p.is_dir()
    ],
    key=lambda p: p.name,
    reverse=True
)

folders_to_delete = run_folders[MAX_OUTPUT_FOLDERS:]

for folder in folders_to_delete:

    print(f"Deleting old output folder: {folder}")

    import shutil

    shutil.rmtree(folder)


if SHOW_TIMING:

    print("\n-----------------------------------")
    print("START TIME :", start_datetime.strftime("%Y-%m-%d %H:%M:%S"))
    print("END TIME   :", end_datetime.strftime("%Y-%m-%d %H:%M:%S"))
    print(f"TOTAL TIME : {elapsed_seconds:,.2f} seconds")
    print(f"TOTAL TIME : {elapsed_minutes:,.2f} minutes")
    print("-----------------------------------")

# Final Cell — Open Output Folder
Open the output folder in Windows Explorer.

In [ ]:
import os

print(f"Opening: {OUTPUT_FOLDER}")

os.startfile(OUTPUT_FOLDER)